# Advanced RAG Techniques — AI Engineering Interview Prep

HyDE, contextual compression, re-ranking, and query expansion.

**Install**: `pip install anthropic sentence-transformers faiss-cpu rank-bm25`

In [ ]:
import anthropic
import os
import numpy as np
import faiss
import re
from sentence_transformers import SentenceTransformer, CrossEncoder

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
MODEL = "claude-haiku-4-5-20251001"
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
DIM = embed_model.get_sentence_embedding_dimension()

def llm(prompt, max_tokens=300):
    resp = client.messages.create(
        model=MODEL, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    return resp.content[0].text

# Knowledge base
corpus = [
    "Transformers use scaled dot-product attention: softmax(QK^T/sqrt(d_k))V.",
    "BERT is trained with masked language modeling: 15% of tokens are masked and predicted.",
    "GPT models use causal (autoregressive) attention, attending only to past tokens.",
    "LoRA injects trainable low-rank matrices into frozen pretrained weights: W + BA.",
    "QLoRA combines 4-bit quantization with LoRA to fit 70B models on a single GPU.",
    "RAG retrieves relevant documents at inference time to augment the LLM context window.",
    "Chunking strategy affects RAG quality: sentence splitter vs fixed-size vs semantic chunking.",
    "Vector databases use ANN algorithms like HNSW for sub-linear similarity search.",
    "Cross-encoders score (query, document) pairs jointly for more accurate re-ranking.",
    "HyDE generates a hypothetical document answer, then embeds it for retrieval.",
    "Contextual compression extracts only relevant passages from retrieved documents.",
    "Multi-query retrieval expands one question into multiple queries for broader coverage.",
]

corpus_embeddings = embed_model.encode(corpus, normalize_embeddings=True).astype(np.float32)
index = faiss.IndexFlatIP(DIM)
index.add(corpus_embeddings)

def dense_search(query, k=5):
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(q_emb, k)
    return [(corpus[i], float(scores[0][j])) for j, i in enumerate(ids[0])]

print(f"Corpus: {len(corpus)} docs")

## 1. HyDE — Hypothetical Document Embeddings

In [ ]:
def hyde_search(query, k=3):
    # Step 1: Generate a hypothetical answer document
    hyde_prompt = f"""Write a short technical passage (2-3 sentences) that would be the ideal answer to:
{query}
Write only the passage, no preamble."""
    hypothetical_doc = llm(hyde_prompt, max_tokens=150)

    # Step 2: Embed the hypothetical doc and search
    hyp_emb = embed_model.encode([hypothetical_doc], normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(hyp_emb, k)
    results = [(corpus[i], float(scores[0][j])) for j, i in enumerate(ids[0])]

    return hypothetical_doc, results

query = "how does parameter efficient fine-tuning work?"

print("=== Standard Dense Search ===")
for doc, score in dense_search(query, k=3):
    print(f"  [{score:.4f}] {doc[:80]}")

print("\n=== HyDE Search ===")
hyp_doc, hyde_results = hyde_search(query)
print(f"Hypothetical doc: {hyp_doc[:150]}...")
print("\nRetrieved:")
for doc, score in hyde_results:
    print(f"  [{score:.4f}] {doc[:80]}")

## 2. Multi-Query Retrieval

In [ ]:
def multi_query_search(query, k=3):
    # Generate multiple query variants
    expand_prompt = f"""Generate 3 different search queries that would retrieve relevant documents for:
Original: {query}

Output ONLY the 3 queries, one per line, no numbering."""
    expanded = llm(expand_prompt, max_tokens=150)
    variants = [q.strip() for q in expanded.strip().split('\n') if q.strip()][:3]
    variants = [query] + variants  # include original

    # Retrieve for each variant, deduplicate with union
    seen = {}
    for q in variants:
        for doc, score in dense_search(q, k=k):
            if doc not in seen or score > seen[doc]:
                seen[doc] = score

    return variants, sorted(seen.items(), key=lambda x: -x[1])[:k]

query = "efficient training of large language models"
variants, results = multi_query_search(query)

print("Query variants generated:")
for v in variants:
    print(f"  - {v}")

print("\nMerged results (union):")
for doc, score in results:
    print(f"  [{score:.4f}] {doc[:80]}")

## 3. Contextual Compression

In [ ]:
# Contextual compression: extract only the relevant sentence from a retrieved chunk
def compress_document(query, document):
    compress_prompt = f"""Extract ONLY the sentence(s) from the document that directly answer the query.
If nothing is relevant, output: NOT_RELEVANT

Query: {query}
Document: {document}

Relevant extract:"""
    return llm(compress_prompt, max_tokens=100)

query = "what is the formula for attention?"
retrieved_docs = [doc for doc, _ in dense_search(query, k=4)]

print(f"Query: '{query}'")
print("\nContextual compression results:")
compressed = []
for doc in retrieved_docs:
    extract = compress_document(query, doc)
    if "NOT_RELEVANT" not in extract:
        compressed.append(extract)
        print(f"  Original: {doc[:60]}...")
        print(f"  Extract:  {extract.strip()[:80]}")
        print()

# Final answer with compressed context
context = "\n".join(compressed)
answer_prompt = f"""Answer the question using only the provided context.

Context:
{context}

Question: {query}"""
answer = llm(answer_prompt, max_tokens=200)
print("Final answer:")
print(answer)

## 4. Re-Ranking with Cross-Encoder

In [ ]:
# Cross-encoder: jointly encodes (query, doc) for precise relevance scoring
try:
    cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    query = "how to efficiently fine-tune large models with limited GPU memory"

    # Step 1: Fast retrieval with bi-encoder (get top-10 candidates)
    candidates = dense_search(query, k=8)

    # Step 2: Re-rank with cross-encoder
    pairs = [[query, doc] for doc, _ in candidates]
    ce_scores = cross_encoder.predict(pairs)

    # Sort by cross-encoder score
    reranked = sorted(zip([doc for doc, _ in candidates], ce_scores),
                      key=lambda x: -x[1])

    print("Bi-encoder ranking:")
    for i, (doc, score) in enumerate(candidates[:5]):
        print(f"  {i+1}. [{score:.4f}] {doc[:70]}")

    print("\nCross-encoder re-ranked:")
    for i, (doc, score) in enumerate(reranked[:5]):
        print(f"  {i+1}. [{score:.3f}] {doc[:70]}")

except Exception as e:
    print(f"Cross-encoder not available: {e}")
    print("Install: pip install sentence-transformers")
    print("Cross-encoder concept: jointly encodes (query, doc) → single relevance score")
    print("Much more accurate than bi-encoder but ~100x slower — use on top-50 candidates only.")

## Interview Tips

- **HyDE**: improves recall when query is short/vague; LLM generates the 'ideal' answer first.
- **Multi-query**: trades cost for recall — each variant retrieves different relevant docs.
- **Contextual compression**: reduces context window usage; pass only the relevant sentence to the LLM.
- **Re-ranking pipeline**: bi-encoder retrieves 50-100 candidates → cross-encoder re-ranks → take top-5.
- **Cross-encoder vs bi-encoder**: cross-encoder is 10-20% more accurate but can't pre-compute embeddings.
- **RAG failure modes**: retrieval failure (relevant doc not retrieved) vs generation failure (LLM ignores context).